# CodeAlpha Task 1: Credit Scoring - Exploratory Data Analysis & Baseline Training

This notebook demonstrates loading the Kaggle *Give Me Some Credit* and *German Credit* datasets, performing exploratory data analysis, plotting correlation coefficients, and fitting a baseline Random Forest classifier.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, roc_curve

# Set plotting style
sns.set_theme(style="whitegrid")

## 1. Load Dataset

In [ ]:
# Resolve relative path to datasets
dataset_path = Path("../datasets/credit_scoring_give_me_some_credit.csv")
if dataset_path.exists():
    df = pd.read_csv(dataset_path)
    print(f"Loaded Give Me Some Credit dataset: {df.shape[0]} rows, {df.shape[1]} columns.")
else:
    # Fallback/mock data for notebook testing if run out-of-tree
    print("Dataset path not found, using synthesized data for notebook demonstration.")
    np.random.seed(42)
    df = pd.DataFrame(np.random.randn(1000, 10), columns=[f'Feature_{i}' for i in range(10)])
    df['SeriousDlqin2yrs'] = np.random.choice([0, 1], size=1000, p=[0.93, 0.07])

df.head()

## 2. Exploratory Data Analysis & Feature Correlations

In [ ]:
plt.figure(figsize=(10, 8))
corr = df.corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", square=True)
plt.title("Correlation Matrix of Credit Features")
plt.show()

## 3. Fit baseline Random Forest Classifier

In [ ]:
# Preprocess data: handle missing values
if 'MonthlyIncome' in df.columns:
    df['MonthlyIncome'] = df['MonthlyIncome'].fillna(df['MonthlyIncome'].median())
if 'NumberOfDependents' in df.columns:
    df['NumberOfDependents'] = df['NumberOfDependents'].fillna(0)

# Target column mapping
target_col = 'SeriousDlqin2yrs' if 'SeriousDlqin2yrs' in df.columns else df.columns[-1]
X = df.drop(columns=[target_col])
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

rf = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob):.4f}")